In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


def plot_efficiency_scatter(csv_path):
    df = pd.read_csv(csv_path)

    def extract_mean(val):
        if isinstance(val, str) and '±' in val:
            return float(val.split('±')[0])
        try:
            return float(val)
        except:
            return 0.0

    df['acc_mean'] = df['acc'].apply(extract_mean)
    df['time'] = df['time(s)'].apply(extract_mean)

    plt.figure(figsize=(9, 7), facecolor='white')

    # 绘制点和连线
    plt.plot(df['time'], df['acc_mean'], linestyle='--', color='#CCCCCC', zorder=1)

    # 循环打点，方便给不同倍率上色
    colors = ['#3498DB', '#27AE60', '#F1C40F', '#E67E22']  # 不同倍率不同颜色
    for i, row in enumerate(df.iterrows()):
        r = row[1]
        plt.scatter(r['time'], r['acc_mean'], s=200, label=r['method'], edgecolors='white', linewidth=1.5, zorder=2)
        plt.text(r['time'] + 1, r['acc_mean'], r['method'], fontsize=11, fontweight='bold')

    # 视觉高亮 10x 区域
    target = df[df['method'].str.contains('10')]
    if not target.empty:
        plt.gca().add_patch(plt.Circle((target['time'].values[0], target['acc_mean'].values[0]),
                                       3, color='#27AE60', fill=False, lw=2, ls='--'))
        plt.text(target['time'].values[0], target['acc_mean'].values[0] + 0.015,
                 'Optimal Trade-off', ha='center', color='#1E8449', fontweight='bold')

    plt.xlabel('Inference Time (s) [Lower is Better]', fontsize=12)
    plt.ylabel('Accuracy [Higher is Better]', fontsize=12)
    plt.title('Performance Pareto Front: Magnification Comparison', size=14, fontweight='bold')
    plt.savefig('mag_point.pdf', dpi=300, bbox_inches='tight')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend(title="Magnification")

    # 自动调整坐标轴范围，给标注留空间
    plt.ylim(df['acc_mean'].min() - 0.05, df['acc_mean'].max() + 0.05)

    plt.tight_layout()
    plt.show()


plot_efficiency_scatter('Mag/all_metrics_summary.csv')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def plot_stain_robustness_radar(csv_path):
    # 1. 数据读取与均值提取
    df = pd.read_csv(csv_path)

    perf_metrics = ['acc', 'macro_recall', 'macro_f1', 'quadratic_kappa', 'macro_pre']
    time_metric = 'time(s)'

    def extract_mean(val):
        if isinstance(val, str) and '±' in val:
            return float(val.split('±')[0])
        try:
            return float(val)
        except:
            return 0.0

    plot_data = df[['method']].copy()
    for m in perf_metrics + [time_metric]:
        plot_data[m] = df[m].apply(extract_mean)

    # 2. 坐标对齐计算 (重点：通过幂运算拉开性能轴差距)
    plot_data['SPEED'] = 1.0 / plot_data[time_metric]

    norm_data = plot_data.copy()

    def amplify_diff(series, power=3.0):
        """
        通过幂运算拉大视觉差距：
        1. 归一化到 0-1
        2. 进行 N 次方处理（高分依然高，稍低的分数会显著向中心收缩）
        3. 重新映射回 0.2-1.0 以保持雷达图的美观
        """
        s_min, s_max = series.min(), series.max()
        if s_max == s_min: return np.ones_like(series)

        # 线性归一化
        linear_norm = (series - s_min) / (s_max - s_min)
        # 幂运算拉开差距，然后映射到 0.2-1.0 范围
        return np.power(linear_norm, power) * 0.5 + 0.55

    for m in perf_metrics:
        power = 2
        if m == 'macro_pre':
            power = 0.5
        norm_data[m] = amplify_diff(plot_data[m], power=power)

    # SPEED 同样处理
    norm_data['SPEED'] = amplify_diff(plot_data['SPEED'], power=3.0)

    # 3. 绘图参数 (完全恢复原样)
    all_metrics = perf_metrics + ['SPEED']
    labels = [l.replace('_', ' ').upper() for l in all_metrics]
    num_vars = len(labels)
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles += angles[:1]

    # 画布初始化
    fig = plt.figure(figsize=(10, 10), facecolor='white')
    ax = fig.add_subplot(111, polar=True)
    ax.set_facecolor('white')

    # 4. 配色方案 (恢复原始参数)
    colormap = plt.cm.get_cmap('Paired', len(df))
    target_method = "h-optimus-1"

    for i, (idx, row) in enumerate(norm_data.iterrows()):
        values = row[all_metrics].values.flatten().tolist()
        values += values[:1]
        name = row['method']

        if name == target_method:
            color = '#B22222'
            lw = 2  # 恢复原始 2
            zorder = 20
        else:
            color = colormap(i)
            lw = 1.8  # 恢复原始 1.8
            zorder = i

        ax.plot(angles, values, linewidth=lw, label=name, color=color, zorder=zorder)

    # 5. 细节美化 (完全恢复原样)
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    plt.xticks(angles[:-1], labels, size=11, fontweight='bold', color='#333333')

    ax.yaxis.grid(True, color="#DDDDDD", linestyle='--', linewidth=0.5)
    ax.xaxis.grid(True, color="#DDDDDD", linewidth=0.8)

    ax.set_ylim(0, 1.1)
    ax.set_yticklabels([])

    ax.spines['polar'].set_visible(False)

    # 6. 图例文字 (完全恢复原样)
    leg = plt.legend(loc='upper right', bbox_to_anchor=(1.25, 1.05),
                     frameon=True, fontsize=10, edgecolor='#EEEEEE')
    for text in leg.get_texts():
        text.set_color("black")

    plt.title('Multi-criteria Evaluation of Stain Normalization',
              size=15, pad=40, fontweight='bold', color='black')
    plt.savefig('stains_trade-off.pdf', dpi=300, bbox_inches='tight')
    plt.tight_layout()
    plt.show()


# 调用
plot_stain_robustness_radar('Stains/all_metrics_summary.csv')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# 1. 数据初始化
cohorts = ['Train/Val', 'Int. Test', 'Ext. 301', 'Ext. YNZL', 'Ext. SL']
patients = [1349, 138, 30, 31, 45]
total_wsis = [2233, 345, 147, 136, 256]

specimen = {
    'CNB': [744, 29, 64, 50, 111],
    'RP': [987, 254, 31, 43, 102],
    'TURP': [502, 62, 52, 43, 43]
}

diagnosis = {
    'Benign': [1096, 50, 119, 63, 71],
    'Malignant': [1137, 295, 28, 73, 166]
}

# 2. 设置绘图风格 (学术简约风)
plt.rcParams['font.family'] = 'Arial'
sns.set_context("paper", font_scale=1.2)
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
plt.subplots_adjust(wspace=0.3, hspace=0.4)

colors_spec = ['#7eb5bc', '#ef9d9a', '#9bb4d5']  # 莫兰迪蓝/粉/灰蓝
colors_diag = ['#82B0D2', '#FA7F6F']

# --- 图 A: Scale Overview (Grouped Bar) ---
x = np.arange(len(cohorts))
width = 0.35
axes[0, 0].bar(x - width / 2, patients, width, label='Patients', color='#D3D3D3', edgecolor='black', linewidth=0.8)
axes[0, 0].bar(x + width / 2, total_wsis, width, label='WSIs', color='#808080', edgecolor='black', linewidth=0.8)
axes[0, 0].set_title('A. Cohort Scale (Patients vs WSIs)', loc='left', fontweight='bold', fontsize=14)
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(cohorts)
axes[0, 0].legend()
axes[0, 0].set_ylabel('Count')

# --- 图 B: Specimen Type (Stacked Bar) ---
bottom = np.zeros(len(cohorts))
for (label, counts), color in zip(specimen.items(), colors_spec):
    axes[0, 1].bar(cohorts, counts, label=label, bottom=bottom, color=color, width=0.6, edgecolor='white')
    bottom += counts
axes[0, 1].set_title('B. Specimen Type Distribution', loc='left', fontweight='bold', fontsize=14)
axes[0, 1].set_ylabel('Number of WSIs')
axes[0, 1].legend(title='Specimen')

# --- 图 C: Diagnosis (Stacked Bar) ---
bottom = np.zeros(len(cohorts))
for (label, counts), color in zip(diagnosis.items(), colors_diag):
    axes[1, 0].bar(cohorts, counts, label=label, bottom=bottom, color=color, width=0.6, edgecolor='white')
    bottom += counts
axes[1, 0].set_title('C. Pathological Diagnosis', loc='left', fontweight='bold', fontsize=14)
axes[1, 0].set_ylabel('Number of WSIs')
axes[1, 0].legend(title='Diagnosis')

# --- 图 D: Normalized Specimen Type (Percentage) ---
# 计算比例
spec_df = pd.DataFrame(specimen)
spec_norm = spec_df.div(spec_df.sum(axis=1), axis=0) * 100
bottom = np.zeros(len(cohorts))
for i, col in enumerate(spec_norm.columns):
    axes[1, 1].bar(cohorts, spec_norm[col], label=col, bottom=bottom, color=colors_spec[i], width=0.6, edgecolor='white')
    bottom += spec_norm[col]
axes[1, 1].set_title('D. Specimen Type Proportion (%)', loc='left', fontweight='bold', fontsize=14)
axes[1, 1].set_ylabel('Percentage (%)')
axes[1, 1].set_ylim(0, 100)

# 3. 统一清理边框
for ax in axes.flat:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='x', rotation=15)

# 4. 导出
plt.tight_layout()
# plt.savefig('Figure1_Data_Detailed.pdf', dpi=300)
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# =========================
# 1. 原始数据
# =========================
data = {
    'Center': ['Train/Val', 'Internal', '301', 'YNZL', 'FJSL'],
    'CNB_B': [307, 2, 50, 24, 39],
    'CNB_M': [437, 27, 14, 26, 72],
    'RP_B': [319, 37, 19, 43, 27],
    'RP_M': [668, 217, 12, 0, 75],
    'TURP_B': [470, 11, 50, 39, 5],
    'TURP_M': [32, 51, 2, 4, 19],
}

df = pd.DataFrame(data)

# =========================
# 2. 计算恶性率与构建标注矩阵
# =========================
specimen_types = ['CNB', 'RP', 'TURP']
heatmap_data = pd.DataFrame(index=df['Center'], columns=specimen_types)
annot_data = []

for idx, row in df.iterrows():
    row_annots = []
    for stype in specimen_types:
        m = row[f'{stype}_M']
        b = row[f'{stype}_B']
        total = m + b
        ratio = (m / total * 100) if total > 0 else np.nan
        heatmap_data.loc[row['Center'], stype] = ratio
        row_annots.append(f"{ratio:.1f}%\n(n={total})")
    annot_data.append(row_annots)

heatmap_data = heatmap_data.astype(float)


sns.set_theme(style="white")
plt.figure(figsize=(7, 5.5))

ax = sns.heatmap(
    heatmap_data,
    annot=np.array(annot_data),
    fmt="",
    cmap="GnBu",          # 核心修改：改为浅青蓝色系
    linewidths=2,         # 增加间距
    linecolor='white',    # 白色网格线
    cbar_kws={'label': 'Malignant Ratio (%)', 'shrink': 0.8}, # 缩小色条更精致
    vmin=0,
    vmax=100,
    annot_kws={"size": 10, "color": "black"} # 文字统一用黑色
)


ax.tick_params(left=False, bottom=False)

plt.xticks(fontsize=10)
plt.yticks(fontsize=10, rotation=0)
plt.savefig('data-cohort-e.svg', dpi=300, bbox_inches='tight')
plt.tight_layout()
plt.show()

In [ ]:
import plotly.graph_objects as go

# 定义节点
label = ["Training/Val", "Internal Test", "External 301", "External YNZL", "External SL",  # 0-4
         "CNB", "RP", "TURP",  # 5-7
         "Benign", "Malignant"]  # 8-9

# 基于表格数据构建流向 (Source, Target, Value)
# 简化逻辑：假设比例在各中心间分布（实际应用中可按真实各中心细分比例调整）
source = [0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 6, 6, 7, 7]
target = [5, 6, 7, 5, 6, 7, 5, 6, 7, 5, 6, 7, 5, 6, 7, 8, 9, 8, 9, 8, 9]
value = [744, 987, 502, 29, 254, 62, 64, 31, 52, 50, 43, 43, 111, 102, 43,  # 中心 -> 类型
         600, 398, 400, 996, 400, 245]  # 类型 -> 诊断 (估算值)

fig = go.Figure(data=[go.Sankey(
    node=dict(pad=15, thickness=20, line=dict(color="black", width=0.5),
              label=label, color="royalblue"),
    link=dict(source=source, target=target, value=value)
)])

fig.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df_bar = pd.DataFrame({
    'Center': ['Train/Val', 'Test', '301', 'YNZL', 'FJSL'],
    'CNB': [744, 29, 64, 50, 111],
    'RP': [987, 254, 31, 43, 102],
    'TURP': [502, 62, 52, 43, 43]
})

df_bar.set_index('Center', inplace=True)
df_perc = df_bar.div(df_bar.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8, 6)) # 稍微缩小宽度使比例更协调

# --- 核心修改：使用更专业的学术配色 ---
# 蓝色：#7294D4 (CNB), 黄色：#F2D06B (RP), 绿色：#7EBC9A (TURP)
colors = ['#7294D4', '#F2D06B', '#7EBC9A']

# 绘制柱状图，添加 edgecolor 使各部分区分更明显
df_perc.plot(kind='bar', stacked=True, ax=ax, color=colors, width=0.7, edgecolor='white', linewidth=0.5)

for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    if height > 5:
        x, y = p.get_xy()
        # 将字体改为灰色 #333333，避免纯黑太突兀
        ax.text(x + width / 2, y + height / 2, f'{height:.1f}%',
                ha='center', va='center', fontsize=9, color='#333333', fontweight='medium')

# 细节美化
plt.ylabel('Composition (%)', fontsize=12, labelpad=10)
plt.xlabel('')
plt.xticks(rotation=0, fontsize=10)
plt.yticks(fontsize=10)

# 优化图例
plt.legend(title='Specimen Type', bbox_to_anchor=(1.02, 1), loc='upper left',
           frameon=False, fontsize=10, title_fontsize=11)

# 移除上方和右侧的边框 (Spines)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.grid(axis='y', linestyle='--', alpha=0.3) # 降低网格线透明度
plt.tight_layout()
plt.savefig('data-cohort-c.svg', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 数据准备
data = {
    'Cohort': ['Train/Val', 'Test', '301', 'YNZL', 'FJSL'],
    'CNB_B': [307, 2, 50, 24, 39], 'CNB_M': [437, 27, 14, 26, 72],
    'RP_B': [319, 37, 19, 43, 27], 'RP_M': [668, 217, 12, 0, 75],
    'TURP_B': [470, 11, 50, 39, 5], 'TURP_M': [32, 51, 2, 4, 19]
}
df = pd.DataFrame(data)

# 颜色配置 (良性: 蓝, 恶性: 红; 饱和度由浅到深: CNB, RP, TURP)
colors = {
    'CNB_B': '#d8e5e7', 'CNB_M': '#f9e2e1', # 极浅莫兰迪
    'RP_B': '#7eb5bc',  'RP_M': '#ef9d9a',  # 莫兰迪原色
    'TURP_B': '#5a8d94', 'TURP_M': '#c87a78' # 降低饱和度的深莫兰迪
}

fig, ax = plt.subplots(figsize=(14, 8), facecolor='white')
ax.set_facecolor('white')

x = np.arange(len(df['Cohort']))
width = 0.25
specimens = ['CNB', 'RP', 'TURP']

# 绘制柱状图并添加数值
for i, spec in enumerate(specimens):
    pos = x + (i - 1) * width
    ax.bar(pos, df[f'{spec}_B'], width, color=colors[f'{spec}_B'], edgecolor='black', linewidth=0.8)
    ax.bar(pos, df[f'{spec}_M'], width, bottom=df[f'{spec}_B'], color=colors[f'{spec}_M'], edgecolor='black', linewidth=0.8)

    # 底部第一层标签: 样本类型
    for j in range(len(x)):
        ax.text(pos[j], -30, spec, ha='center', va='top', fontsize=9, color='black', fontweight='bold')
        # 柱子顶部数值
        total = df[f'{spec}_B'][j] + df[f'{spec}_M'][j]
        if total > 0:
            ax.text(pos[j], total + 5, str(int(total)), ha='center', va='bottom', fontsize=8, color='black')

ax.set_xticks(x)
ax.set_xticklabels(df['Cohort'], fontsize=11, fontweight='bold', color='black', y=-0.08)

# 样式美化
ax.set_ylabel('Number of WSIs', fontsize=12, fontweight='bold', color='black')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_color('black')
ax.tick_params(axis='y', colors='black')
ax.tick_params(axis='x', length=0) # 隐藏刻度线，通过位置区分


from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#7eb5bc', edgecolor='black', label='Benign'),
    Patch(facecolor='#ef9d9a', edgecolor='black', label='Malignant')
]
ax.legend(handles=legend_elements, loc='upper right', frameon=True,
          facecolor='white', edgecolor='black', fontsize=10, title="Diagnosis")
plt.savefig('data-cohort-a.svg', bbox_inches='tight', dpi=300)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

data = {
    'Center': ['Train/Val', 'Test', '301', 'YNZL', 'FJSL'],
    'Benign': [1096, 50, 119, 63, 71],
    'Malignant': [1137, 295, 28, 73, 166],
    'Patients': [1349, 138, 30, 31, 45] # 对应各中心病人数 [cite: 1, 31]
}
df = pd.DataFrame(data)

# 2. 设置浅色系参数
color_benign = '#A9CCE3'  # 柔和浅蓝
color_malignant = '#F5B7B1'  # 柔和浅粉/珊瑚色
bar_height = 0.65  # 稍微加宽条形，方便容纳文字


def draw_horizontal_distribution_light():
    fig, ax = plt.subplots(figsize=(13, 7), dpi=300) # 略微加宽 figsize 确保右侧文字显示完整
    y_pos = np.arange(len(df['Center']))

    # 3. 绘制条形图
    ax.barh(y_pos, df['Benign'], bar_height, label='Benign',
            color=color_benign, edgecolor='white', linewidth=1.5)

    ax.barh(y_pos, df['Malignant'], bar_height, left=df['Benign'], label='Malignant',
            color=color_malignant, edgecolor='white', linewidth=1.5)

    # 4. 智能添加标注：百分比 + 具体数量 + 病人数
    for i in range(len(df)):
        total_wsi = df.iloc[i]['Benign'] + df.iloc[i]['Malignant']
        pts = df.iloc[i]['Patients']
        b_val = df.iloc[i]['Benign']
        m_val = df.iloc[i]['Malignant']
        b_pct = b_val / total_wsi * 100
        m_pct = m_val / total_wsi * 100

        # 在右侧标注该中心总样本数和病人数 [cite: 1, 31]
        ax.text(total_wsi + 40, i, f'Total: {total_wsi} WSIs\n(n = {pts} pts)', va='center',
                ha='left', fontweight='bold', fontsize=10, color='#555555')

        if b_val > 80:
            label_b = f'{b_pct:.1f}%\n({b_val})'
            ax.text(b_val / 2, i, label_b, va='center', ha='center',
                    color='#2C3E50', fontsize=9, fontweight='bold')
        else:
            ax.text(b_val / 2, i, f'{b_val}', va='center', ha='center', color='#2C3E50', fontsize=8)

        # --- 内部标注: Malignant ---
        if m_val > 80:
            label_m = f'{m_pct:.1f}%\n({m_val})'
            ax.text(b_val + m_val / 2, i, label_m, va='center', ha='center',
                    color='#2C3E50', fontsize=9, fontweight='bold')
        else:
            ax.text(b_val + m_val / 2, i, f'{m_val}', va='center', ha='center', color='#2C3E50', fontsize=8)

    # 5. 样式美美化
    ax.set_yticks(y_pos)
    ax.set_yticklabels(df['Center'], fontsize=12, fontweight='medium')
    ax.invert_yaxis()

    # 移除多余边框
    for spine in ['top', 'right', 'left']:
        ax.spines[spine].set_visible(False)

    ax.legend(loc='lower right', frameon=True, fontsize=10)

    # 调整范围
    ax.set_xlim(0, 2800) # 调大上限以容纳新加入的文字
    ax.grid(axis='x', linestyle=':', alpha=0.4, zorder=0)

    plt.tight_layout()
    plt.savefig('data-cohort-d.svg', bbox_inches='tight', dpi=300)
    plt.show()


if __name__ == "__main__":
    draw_horizontal_distribution_light()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

data = {
    'Center': ['Train/Val', 'Internal Test', '301 Hospital', 'YNZL Hospital', 'FJSL Hospital'],
    'CNB': [744, 29, 64, 50, 111],
    'RP': [987, 254, 31, 43, 102],
    'TURP': [502, 62, 52, 43, 43],
    'MainColor': ['#084594', '#4292c6', '#ef3b2c', '#41ab5d', '#807dba']
}
df = pd.DataFrame(data)

if not os.path.exists('donut_charts'):
    os.makedirs('donut_charts')

for i, row in df.iterrows():
    counts = [row['CNB'], row['RP'], row['TURP']]
    labels = ['CNB', 'RP', 'TURP']
    total = sum(counts)
    # 颜色梯度
    center_colors = [row['MainColor'], row['MainColor'] + 'AA', row['MainColor'] + '66']

    fig, ax = plt.subplots(figsize=(5, 5), dpi=300)

    wedges, _ = ax.pie(
        counts,
        colors=center_colors,
        startangle=90,
        wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2),
        counterclock=False
    )

    for j, p in enumerate(counts):
        percentage = (p / total) * 100
        center_ang = (wedges[j].theta1 + wedges[j].theta2) / 2.
        rad = np.deg2rad(center_ang)

        # 锚点在圆环中心
        x, y = 0.8 * np.cos(rad), 0.8 * np.sin(rad)

        x_text = 1.15 * np.sign(x)
        y_text = 1.1 * y

        ha = "left" if x > 0 else "right"
        display_text = f"{labels[j]}\nn={p}\n({percentage:.1f}%)"

        ax.annotate(
            display_text,
            xy=(x, y),
            xytext=(x_text, y_text),
            horizontalalignment=ha,
            verticalalignment='center',
            fontsize=16,  # 稍微调小字号配合紧凑布局
            fontweight='bold',
            arrowprops=dict(
                arrowstyle='-',
                color=row['MainColor'],
                linewidth=1.2,
                connectionstyle=f"angle,angleA=0,angleB={center_ang}",
                shrinkB=2
            )
        )

    plt.title(row['Center'], fontsize=14, fontweight='bold', pad=5, color=row['MainColor'])

    ax.set(xlim=(-1.3, 1.3), ylim=(-1.3, 1.3))

    file_name = row['Center'].replace('/', '_').replace(' ', '_')
    plt.savefig(f"donut_charts/{file_name}_donut.png", transparent=True, bbox_inches='tight')
    plt.show()
    plt.close()

print("调整完成！标题与标注已向圆环靠拢。")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 数据准备
data = {
    'Cohort': ['Train/Val', 'Test', '301', 'YNZL', 'FJSL'],
    'CNB_B': [307, 2, 50, 24, 39], 'CNB_M': [437, 27, 14, 26, 72],
    'RP_B': [319, 37, 19, 43, 27], 'RP_M': [668, 217, 12, 0, 75],
    'TURP_B': [470, 11, 50, 39, 5], 'TURP_M': [32, 51, 2, 4, 19]
}
df = pd.DataFrame(data)

colors = {
    'CNB_B': '#DEEBF7', 'CNB_M': '#FEE0D2',
    'RP_B': '#9ECAE1',  'RP_M': '#FC9272',
    'TURP_B': '#3182BD', 'TURP_M': '#DE2D26'
}

fig, ax = plt.subplots(figsize=(14, 8), facecolor='white')
ax.set_facecolor('white')

x = np.arange(len(df['Cohort']))
width = 0.25
specimens = ['CNB', 'RP', 'TURP']

for i, spec in enumerate(specimens):
    pos = x + (i - 1) * width
    ax.bar(pos, df[f'{spec}_B'], width, color=colors[f'{spec}_B'], edgecolor='black', linewidth=0.8)
    ax.bar(pos, df[f'{spec}_M'], width, bottom=df[f'{spec}_B'], color=colors[f'{spec}_M'], edgecolor='black', linewidth=0.8)

    for j in range(len(x)):
        ax.text(pos[j], -30, spec, ha='center', va='top', fontsize=9, color='black', fontweight='bold')
        total = df[f'{spec}_B'][j] + df[f'{spec}_M'][j]
        if total > 0:
            ax.text(pos[j], total + 5, str(int(total)), ha='center', va='bottom', fontsize=8, color='black')

ax.set_xticks(x)
ax.set_xticklabels(df['Cohort'], fontsize=11, fontweight='bold', color='black', y=-0.08)

ax.set_ylabel('Number of WSIs', fontsize=12, fontweight='bold', color='black')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_color('black')
ax.spines['bottom'].set_color('black')
ax.tick_params(axis='y', colors='black')
ax.tick_params(axis='x', length=0)


from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3182BD', edgecolor='black', label='Benign (Blue)'),
    Patch(facecolor='#DE2D26', edgecolor='black', label='Malignant (Red)')
]
ax.legend(handles=legend_elements, loc='upper right', frameon=True,
          facecolor='white', edgecolor='black', fontsize=10, title="Diagnosis")
plt.savefig('data-cohort-a.svg')
plt.tight_layout()
plt.show()